In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tabulate import tabulate 

# -----------------------------
# Dataset classes
# -----------------------------
class MatrixDataset(Dataset):
    def __init__(self, matrices, labels, file_paths=None):
        self.matrices = matrices
        self.labels = labels
        self.file_paths = file_paths if file_paths is not None else [None] * len(matrices)

    def __len__(self):
        return len(self.matrices)

    def __getitem__(self, idx):
        mat = torch.tensor(self.matrices[idx], dtype=torch.float32)
        mat = (mat - mat.mean()) / (mat.std() + 1e-6)  # normalize
        mat = mat.unsqueeze(0)  # [1, H, W] for CNN
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        path = self.file_paths[idx]
        return mat, label, path


# -----------------------------
# CNN Model
# -----------------------------
class CNN(nn.Module):
    def __init__(self, num_classes, input_shape):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Figure out flattened size by running dummy input
        with torch.no_grad():
            dummy = torch.zeros(1, 1, *input_shape)  # (B, C, H, W)
            out = self._forward_features(dummy)
            flat_size = out.view(1, -1).size(1)

        self.fc1 = nn.Linear(flat_size, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def _forward_features(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        return x

    def forward(self, x):
        x = self._forward_features(x)
        x = x.view(x.size(0), -1)  # flatten
        x = F.relu(self.fc1(x))
        return self.fc2(x)

print("done")

done


In [2]:
# -----------------------------
# Load Data
# -----------------------------
train = np.loadtxt("../fiot_highway2-main/train.txt", dtype=str).tolist()
test  = np.loadtxt("../fiot_highway2-main/test.txt",  dtype=str).tolist()
print(1)

# Split into paths and classes
train_file_paths = [row[0] for row in train]
test_file_paths  = [row[0] for row in test]
print(2)

train_labels     = [int(row[1]) for row in train]
test_labels      = [int(row[1]) for row in test]
print(3)

train_limit = int(len(train_file_paths) * 0.25)
test_limit  = int(len(test_file_paths) * 0.25)
print(4)

train_matrixes  = [np.load("../fiot_highway2-main/" + fp) for fp in train_file_paths[:train_limit]]
test_matrixes   = [np.load("../fiot_highway2-main/" + fp) for fp in test_file_paths[:test_limit]]

print(5)

# Also slice the labels and paths accordingly
train_labels     = train_labels[:train_limit]
test_labels      = test_labels[:test_limit]
train_file_paths = train_file_paths[:train_limit]
test_file_paths  = test_file_paths[:test_limit]
print("done")

1
2
3
4
5
done


In [3]:
H, W = 512, 243
num_classes = 9  # labels 0–8

# Limit dataset for quick testing (remove [:int(...)] if you want full dataset)
train_dataset = MatrixDataset(
    train_matrixes,
    train_labels,
    train_file_paths
)

test_dataset = MatrixDataset(
    test_matrixes,
    test_labels,
    test_file_paths
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)
print("done")

done


In [4]:
# -----------------------------
# Training
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
print(1)
model = CNN(num_classes=num_classes, input_shape=(H, W)).to(device)
print(2)
criterion = nn.CrossEntropyLoss()
print(3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print("done")

1
2
3
done


In [5]:
# -----------------------------
# Training Loop
# -----------------------------

for epoch in range(5):
    print(epoch)
    # Training loop
    model.train()
    running_loss = 0.0
    for X, y, _ in train_loader:
        X, y = X.to(device), y.to(device)

        outputs = model(X)
        loss = criterion(outputs, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        print(f"\rEvaluating {running_loss}", end="")

    avg_loss = running_loss / len(train_loader)

    # -----------------------------
    # Evaluation
    # -----------------------------
    model.eval()
    correct, total = 0, 0
    rows = []
    
    with torch.no_grad():
        for batch_idx, (X, y, paths) in enumerate(test_loader):  # now includes paths
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            _, preds = torch.max(outputs, 1)
    
            for i in range(len(y)):
                filepath = paths[i]                       # e.g. "000000.npy"
                true_class = y[i].item()
                pred_class = preds[i].item()
                is_correct = (true_class == pred_class)
                rows.append([filepath, true_class, pred_class, is_correct])
    
            correct += (preds == y).sum().item()
            total += y.size(0)
    
            # One-line progress update
            print(f"\rEvaluating batch {batch_idx+1}/{len(test_loader)}", end="")
    
    acc = correct / total * 100
    print(f"\nEpoch {epoch+1}, Loss: {avg_loss:.4f}, Test Accuracy: {acc:.2f}%")
    
    # Print table for current epoch
    headers = ["File", "True", "Pred", "Correct"]
    print(tabulate(rows, headers=headers))

0
Evaluating batch 52/522859497
Epoch 1, Loss: 1.4521, Test Accuracy: 65.05%
File               True    Pred  Correct
---------------  ------  ------  ---------
data/012133.npy       8       2  False
data/012821.npy       8       2  False
data/008239.npy       7       0  False
data/016011.npy       5       5  True
data/014592.npy       8       2  False
data/002010.npy       7       0  False
data/000764.npy       5       5  True
data/008235.npy       7       0  False
data/002803.npy       7       0  False
data/000259.npy       3       0  False
data/000163.npy       3       0  False
data/003961.npy       5       5  True
data/009167.npy       7       0  False
data/000355.npy       3       0  False
data/004640.npy       5       5  True
data/011895.npy       7       0  False
data/002814.npy       7       0  False
data/006583.npy       1       2  False
data/009839.npy       1       0  False
data/012140.npy       8       2  False
data/005752.npy       5       5  True
data/003531.npy       1  